In [1]:
import json
from pathlib import Path

import requests
import pandas as pd

# import altair as alt
# import lxml

In [2]:
league_id = 'kfszz7vdmdl0krou'
season = '15'
latest_gw = 23

In [3]:
records = []

for period in range(1, 39):  # periods 1–38
    url = f'https://www.fantrax.com/fxpa/req?leagueId={league_id}'

    headers = {
        "accept": "application/json",
        "content-type": "text/plain",
        "sec-ch-ua": "\"Not;A=Brand\";v=\"99\", \"Google Chrome\";v=\"139\", \"Chromium\";v=\"139\"",
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": "\"macOS\"",
        "referer": f"https://www.fantrax.com/fantasy/league/{league_id}/standings;view=REGULAR_SEASON;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period=38"
    }
    
    payload = {
        "msgs": [
            {
                "method": "getStandings",
                "data": {
                    "leagueId": league_id,
                    "view": "REGULAR_SEASON",
                    "timeframeType": "BY_PERIOD",
                    "timeStartType": "FROM_SEASON_START",
                    "period": str(period)
                }
            }
        ],
        "uiv": 3,
        "refUrl": f"https://www.fantrax.com/fantasy/league/{league_id}/standings;view=REGULAR_SEASON;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period={period}",
        "dt": 1,
        "at": 0,
        "av": "0.0",
        "tz": "America/Los_Angeles",
        "v": "179.0.1"
    }

    r = requests.post(url, headers=headers, json=payload)
    r.raise_for_status()
    j = r.json()    

    data = j["responses"][0]["data"]
    team_info = data.get("fantasyTeamInfo", {})

    # find the "Standings" table
    standings_tbl = next(
        (t for t in data.get("tableList", []) if t.get("caption") == "Standings"),
        None
    )
    if not standings_tbl:
        print(f"Period {period}: Standings table not found")
        continue

    for row in standings_tbl.get("rows", []):
        fixed = row.get("fixedCells", [])
        cells = row.get("cells", [])

        # fixed cells: [rank, team cell]
        rank = fixed[0].get("content") if len(fixed) > 0 else None
        team_cell = fixed[1] if len(fixed) > 1 else {}
        team_name = team_cell.get("content")
        team_id = team_cell.get("teamId")

        ti = team_info.get(team_id, {})
        record = {
            "gw": period,
            "rank": rank,
            "teamId": team_id,
            "team": team_name,
            "shortName": ti.get("shortName"),
            "logo": ti.get("logoUrl512"),
            # header order for cells: W, D, L, Points, Win%, WW, FPtsF, FPtsA, Streak
            "W": cells[0].get("content") if len(cells) > 0 else None,
            "D": cells[1].get("content") if len(cells) > 1 else None,
            "L": cells[2].get("content") if len(cells) > 2 else None,
            "Points": cells[3].get("content") if len(cells) > 3 else None,
            "Win%": cells[4].get("content") if len(cells) > 4 else None,
            "WW": cells[5].get("content") if len(cells) > 5 else None,
            "FPtsF": cells[6].get("content") if len(cells) > 6 else None,
            "FPtsA": cells[7].get("content") if len(cells) > 7 else None,
            "Streak": cells[8].get("content") if len(cells) > 8 else None,
        }
        records.append(record)

df = pd.DataFrame(records)

# make numeric columns numeric
for col in ["rank", "W", "D", "L", "Points", "Win%", "WW", "FPtsF", "FPtsA"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [4]:
df.shortName.unique()

array(['BenMtz', 'MT', 'NU', 'TJ', 'CFC', 'SHFC', 'MBruno', 'THHS', 'Zac',
       'APSB'], dtype=object)

In [5]:
df_trim = df[
    df.gw <= latest_gw
][['W', 'D', 'L','Points','Win%', 'WW', 'FPtsF', 'FPtsA', 'Streak','team','rank','gw']]

In [6]:
# data = df_trim

# chart = alt.Chart(data).mark_line(point=True).encode(
#     x = alt.X('gw', scale=alt.Scale(domain=[1, latest_gw]), title="Gameweek"),
#     y=alt.Y('rank', aggregate={'argmax': 'gw'}, scale=alt.Scale(domain=[10,1]), title="Rank"),
#     color=alt.Color("team", legend=None),
# ).transform_window(
#     rank="rank()",
#     sort=[alt.SortField("rank", order="ascending")],
#     groupby=["gw"]
# ).properties(
#     title="Fake Internet Soccer XIV standings by gameweek",
#     width=700,
#     height=350,
# )

# labels = alt.Chart(data).mark_text(
#     align='left', dx=5
# ).encode(
#     x = alt.X('max(gw)', scale=alt.Scale(domain=[1, latest_gw])),
#     y=alt.Y('rank', aggregate={'argmax': 'gw'}, scale=alt.Scale(domain=[10,1])),
#     text='team:N',
#     color='team:N',
# ).transform_window(
#     rank="rank()",
#     sort=[alt.SortField("rank", order="ascending")],
#     groupby=["gw"]
# )

# chart + labels

In [7]:
df_trim.to_csv(f"data/output/standings/standings-season-{season}.csv", index=False)

In [8]:
latest_gw = df_trim[df_trim.gw == df_trim.gw.max()]

In [9]:
pivot_df = pd.pivot(df_trim, index='gw', columns='team', values='rank').reset_index()
pivot_df.to_csv(f'data/output/datawrapper/season-standings-{season}.csv', index=False)

In [10]:
access_token = 'yXNpSzS8XN1dCvPgQUS2ofNz3IvncMH26TkRLZe2stW0fCDxXZAnCufjAOfRNiDc'
chart_id = 'b7v86'
base_url = 'https://api.datawrapper.de/v3/charts'

chart_url = f'{base_url}/{chart_id}'
chart_data_url = f'{chart_url}/data'
publish_url = f'{chart_url}/publish'

json_headers={
    'accept': '*/*',
    'Authorization':f'Bearer {access_token}',
    'content-type':'application/json' 
}
csv_headers={
    'accept': '*/*',
    'Authorization':f'Bearer {access_token}',
    'content-type':'text/csv'
}

In [11]:
update_data_response = requests.put(chart_data_url, data=pivot_df.to_csv(index=False), headers=csv_headers)
update_data_response

<Response [204]>

In [12]:
ranks_dict = latest_gw[['team','rank']].set_index('team')['rank'].to_dict()

In [13]:
chart_lines = {}

for team, rank in ranks_dict.items():
    symbol = "square" if rank % 2 == 0 else "circle"
    print(f"{team}: {symbol}")
    line_config = {
        "symbols": {
            "style": "hollow",
            "enabled": True,
            "shape": symbol
        }
    }

    chart_lines[team] = line_config

Mattchester United: circle
GAK-PO-TAY-TOES: square
Magpies United: circle
Cheasle FC: square
Benford FC: circle
Morning Timber: square
Seanhampton: circle
FPL 5: AutoPick Strikes Back: square
Thottenham Hotsluts: circle
Walton Goggonzola: square


In [14]:
chart_lines

{'Mattchester United': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'GAK-PO-TAY-TOES': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}},
 'Magpies United': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'Cheasle FC': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}},
 'Benford FC': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'Morning Timber': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}},
 'Seanhampton': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'FPL 5: AutoPick Strikes Back': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}},
 'Thottenham Hotsluts': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'Walton Goggonzola': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}}}

In [15]:
payload = {
    "metadata": {
        "visualize": {
            "lines": chart_lines,
            "color-category": {
                "map": {
                    "Walton Goggonzola": "#332288",  # indigo
                    "Cheasle FC": "#88CCEE",        # cyan
                    "Liverwirtz Football Club": "#44AA99",  # teal
                    "Seanhampton": "#117733",       # green
                    "Benford FC": "#999933",        # olive
                    "Morning Timber": "#DDCC77",    # sand
                    "Mattchester United": "#CC6677",# rose
                    "Magpies United": "#882255",    # wine
                    "Thottenham Hotsluts": "#AA4499",# purple
                    "FPL 5: AutoPick Strikes Back": "#DDDDDD" # pale grey
                }
            }
        }
    }
}

In [16]:
patch_chart_response = requests.patch(chart_url, json=payload, headers=json_headers)
patch_chart_response

<Response [200]>

In [17]:
publish_chart_response = requests.post(publish_url,headers=json_headers)
publish_chart_response

<Response [200]>

In [18]:
latest_chart_version = publish_chart_response.json()['data']['publicVersion']
latest_chart_version

20

In [19]:
filepath = Path('../_data/charts.json')
with filepath.open("r", encoding="utf-8") as f:
    charts = json.load(f)

charts[season]['standings']['version'] = latest_chart_version

with filepath.open("w", encoding="utf-8") as f:
    json.dump(charts, f, indent=2)